# map/ — Colab runner (Parametric UMAP / TensorFlow)

Chạy pipeline bản đồ trên Colab — đặc biệt `pumap2d` (Parametric UMAP cần TensorFlow, local mac crash).
Code clone từ GitHub; **data đặt sẵn trên Drive** ở `MyDrive/map_data/`; HTML ghi thẳng về Drive.

## Đặt các folder này vào `MyDrive/map_data/` (1 lần)
```
MyDrive/map_data/
├── artifacts/            # item_vectors.npy, item_index.parquet, user_tower.pt,
│                         #   users_history.parquet, user_split.parquet  (copy cả folder artifacts/ cũng được)
├── train-data/           # feature_spec.json, item_features.parquet,
│                         #   synopsis_emb.npy, synopsis_low_info.npy   (cho genre-probe qua best.pt)
├── checkpoints/          # best.pt
└── cleaned-data/         # details.csv
```
Notebook tự symlink các folder này vào đúng path repo + symlink `map/outputs/` → `MyDrive/map_data/outputs/`
(coords/reducer/HTML đều persist về Drive; kéo HTML về local mở browser).

In [ ]:
# 0) Mount Drive + clone code từ GitHub + cd vào repo
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

MAP_DATA = '/content/drive/MyDrive/map_data'
assert os.path.isdir(MAP_DATA), f'Khong thay {MAP_DATA} — tao folder + upload data (xem cell tren).'

REPO   = 'https://github.com/CrocodixD/anime-recommender.git'
BRANCH = 'main'            # đổi nếu push code map/ ở branch khác
CODE   = '/content/recommender'

if os.path.exists(CODE):
    !cd {CODE} && git fetch origin && git checkout {BRANCH} && git pull
else:
    !git clone -b {BRANCH} {REPO} {CODE}

%cd {CODE}

In [ ]:
# 1) Symlink data trên Drive vào đúng path repo (code dùng path tương đối từ ROOT repo)
import os
links = {
    f'{MAP_DATA}/artifacts':   f'{CODE}/artifacts',
    f'{MAP_DATA}/train-data':  f'{CODE}/retriever/train-data',
    f'{MAP_DATA}/checkpoints': f'{CODE}/retriever/checkpoints',
    f'{MAP_DATA}/cleaned-data':f'{CODE}/cleaned-data',
}
for src, dst in links.items():
    assert os.path.isdir(src), f'Thieu {src} tren Drive'
    if os.path.islink(dst) or os.path.exists(dst):
        if os.path.islink(dst): os.unlink(dst)
    os.symlink(src, dst)
    print('link', dst, '->', src)

# outputs/ symlink ra Drive => coords/reducer/HTML persist + kéo về local
os.makedirs(f'{MAP_DATA}/outputs', exist_ok=True)
out = f'{CODE}/map/outputs'
if os.path.islink(out) or os.path.exists(out):
    if os.path.islink(out): os.unlink(out)
os.symlink(f'{MAP_DATA}/outputs', out)
print('link', out, '-> Drive/outputs')

In [ ]:
# 2) Cài lib map/ còn thiếu trên Colab (torch/tensorflow/sklearn/plotly/pyarrow đã có sẵn)
!pip install -q umap-learn pacmap polars joblib
import tensorflow as tf, torch
print('tf', tf.__version__, '| torch', torch.__version__)   # TF chạy OK trên Colab (khác mac local)

In [ ]:
# 3) Sanity: file đầu vào đã link đủ chưa
import os
need = ['artifacts/item_vectors.npy','artifacts/item_index.parquet','artifacts/user_tower.pt',
        'artifacts/users_history.parquet','artifacts/user_split.parquet',
        'retriever/train-data/feature_spec.json','retriever/train-data/item_features.parquet',
        'retriever/train-data/synopsis_emb.npy','retriever/train-data/synopsis_low_info.npy',
        'retriever/checkpoints/best.pt','cleaned-data/details.csv']
miss = [f for f in need if not os.path.exists(f)]
print('THIEU:', miss) if miss else print('OK — đủ file đầu vào.')

In [ ]:
# 4) Base 1 lần (join vector + metadata genre)
!python map/build_base.py

In [ ]:
# 5) Fit projection — pumap2d là điểm chính (TF). Thêm umap_sphere/2d/pca để so sánh.
METHODS = ['pumap2d', 'umap2d', 'umap_sphere', 'pca2d', 'pca_sphere']
for m in METHODS:
    print(f'\n===== project {m} =====')
    !python map/project.py --method {m}

In [ ]:
# 6) Cluster 128-d (tô màu chéo mọi map)
!python map/cluster.py --algo kmeans --k 20

In [ ]:
# 7) Đặt điểm mới lên map (chọn method có reducer). USERNAME = user trong dataset.
#    (in vài username mẫu nếu chưa biết)
import pandas as pd
s = pd.read_parquet('artifacts/user_split.parquet')
print('vài username mẫu:', [u for u in s['username'] if u[0].isalnum()][:5])

USERNAME = ''                       # <-- điền 1 username (hoặc để trống và dùng --mal-ids)
for m in ['pumap2d', 'umap_sphere']:
    if USERNAME:
        !python map/encode_user.py "{USERNAME}" --method {m}
    !python map/encode_genre.py --method {m} --all

In [ ]:
# 8) Render HTML tương tác -> ghi thẳng về Drive (map/outputs symlink)
OVERLAYS = ['overlay_genre_all'] + ([f'overlay_user_{USERNAME}'] if USERNAME else [])
ov = ' '.join(f'"{o}"' for o in OVERLAYS)
!python map/viz.py --method pumap2d   --color cluster --cluster kmeans --overlay {ov} --suffix demo
!python map/viz.py --method umap_sphere --color primary_genre --overlay {ov} --suffix demo

In [ ]:
# 9) Xác nhận HTML đã nằm trên Drive (kéo về local mở browser)
!ls -lhS /content/drive/MyDrive/map_data/outputs/*.html